In [0]:
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/adb_medsync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# dbutils.fs.ls raises an exception if the path (or any parent directory) does not exist
try:
    landing_files = dbutils.fs.ls(landing_path)
    path_has_files = len(landing_files) > 0
except Exception:
    path_has_files = False

# Exit outside the try block so the exit signal is not swallowed by the except clause
if not path_has_files:
    msg = f"No new files received at {landing_path}. Exiting notebook."
    print(msg)
    dbutils.notebook.exit(msg)

print(f"Found {len(landing_files)} item(s) at {landing_path}")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_medsync.bronze.{table_name}")